# TP 3 — Simulation de variables aléatoires réelles à densité

**Bibliothèques :** `numpy`, `scipy`, `matplotlib`, `seaborn`. Si nécessaire, dans une cellule Jupyter : `%pip install numpy scipy matplotlib seaborn`.


### Histogramme normalisé et densité estimée (KDE)

Un histogramme regroupe les observations en intervalles. Une **estimation de densité par noyau** (*Kernel Density Estimation*, **KDE**) lisse les observations pour obtenir une **courbe de densité estimée**.

Avec Seaborn, une seule commande permet de tracer les deux :

```python
sns.histplot(donnees, bins=30, stat="density", kde=True)
```

- `stat="density"` : l'aire totale de l'histogramme vaut 1 ;
- `kde=True` : ajoute la **densité estimée par noyau gaussien** 

**Pour ceux qui veulent comprendre la méthode KDE :** [Seaborn — Visualizing distributions of data, section « Kernel density estimation »](https://seaborn.pydata.org/tutorial/distributions.html#kernel-density-estimation). Voir aussi [SciPy — Kernel density estimation](https://docs.scipy.org/doc/scipy/tutorial/stats/kernel_density_estimation.html).

> **Attention :** près d'une borne imposée (par exemple une variable positive ou une loi tronquée), une KDE gaussienne peut présenter des effets de bord. Nous l'utilisons ici comme outil de visualisation.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm

---
## Exercice 1 — Loi normale et transformations

**a)** Simuler un échantillon de **1 000 observations** d'une loi $\mathcal N(0,1)$. Tracer son **histogramme normalisé** et la **courbe KDE**, puis superposer la **densité théorique** sur le même graphique. Comparer les deux courbes.

**Rappel Python :** `norm.pdf(t, loc=0, scale=1)` calcule la densité normale aux points `t`.

**b)** À partir de l'échantillon précédent, construire un échantillon de $Y=1+2X$. Tracer son **histogramme normalisé et sa courbe KDE**. **Quelle loi obtient-on ?**

**c)** Construire de même un échantillon de $W=e^X$, puis tracer son **histogramme normalisé et sa courbe KDE**. Cette loi, appelée **loi log-normale**, a-t-elle une forme symétrique ?

**Question a:**

In [ ]:
# Exercice 1a — compléter les ...
# help(stats.norm)
n=1000
x_norm = stats.norm.rvs(...)

grille = np.linspace(-4, 4, 400)
plt.figure(figsize=(7, 4))
sns.histplot(x_norm, bins=30, stat=..., kde=...,
             alpha=0.40, color="steelblue", label="Histogramme + KDE")
plt.plot(grille, norm.pdf(grille, loc=0, scale=1), color="black",
         linestyle="--", lw=2, label="Densité théorique N(0,1)")
plt.xlabel("x")
plt.ylabel("Densité")
plt.title("Normale centrée réduite : histogramme, KDE et densité théorique")
plt.legend()
plt.show()

**Questions b et c :** 


In [ ]:
# Exercice 1b — transformation affine
Y = ...

# Exercice 1c — transformation exponentielle
W = np.exp(...)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
sns.histplot(Y, bins=30, stat="density", kde=True, ax=axes[0], color="steelblue", alpha=0.45)
axes[0].set(title="Y = 1 + 2X", xlabel="y", ylabel="Densité")
sns.histplot(W, bins=30, stat="density", kde=True, ax=axes[1], color="steelblue", alpha=0.45)
axes[1].set(title="W = exp(X)", xlabel="w", ylabel="Densité")
plt.show()

---
## Exercice 2 — Simulation par rejet : lois tronquées

Pour simuler une loi **tronquée sur $[a,b]$**, on utilise la méthode de rejet :

1. **Tirer** une valeur dans la loi initiale.
2. **Tester** si elle appartient à $[a,b]$.
3. **Conserver** la valeur si elle appartient à l'intervalle ; sinon, la rejeter.
4. **Recommencer** jusqu'à obtenir les $1\,000$ valeurs souhaitées.

**a)** Simuler par rejet **1 000 observations** d'une loi $\mathcal N(0,1)$ tronquée à $[-1,1]$. Tracer l'**histogramme normalisé et la courbe KDE**, puis les comparer à la densité tronquée

$$
\boxed{g(t)=\frac{\varphi(t)}{\Phi(1)-\Phi(-1)}\,\mathbf 1_{[-1,1]}(t),}
$$

où $\varphi$ désigne la densité normale centrée réduite et $\Phi$ sa fonction de répartition (`norm.pdf` et `norm.cdf`).

**b)** Reprendre **la même logique de tirage–test–conservation** pour une **loi exponentielle de paramètre $\lambda=1$**, tronquée à $[0.5,2]$. Tracer son **histogramme normalisé et sa courbe KDE**.

**Rappel Python :** `stats.expon.rvs(scale=1/lambda)` tire dans la loi exponentielle de **taux** $\lambda$. Le paramètre `scale` correspond à $1/\lambda$ ([documentation SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.expon.html)).

> La courbe KDE est une estimation lissée ; près des bornes d'une loi, elle peut dépasser légèrement ou présenter un effet de bord.

In [ ]:
# Exercice 2a — normale tronquée sur [-1, 1]
x_tronq_norm = []

while len(x_tronq_norm) < n:
    tirage = stats.norm.rvs(loc=0, scale=1)
    if ...:   
        x_tronq_norm.append(tirage)

x_tronq_norm = np.array(x_tronq_norm)

# Histogramme + KDE + densité tronquée théorique
t = np.linspace(-1, 1, 350)
g = norm.pdf(t) / ...

plt.figure(figsize=(7, 4))
sns.histplot(x_tronq_norm, bins=25, stat="density", kde=True,
             kde_kws={"cut": 0, "clip": (-1, 1)},
             alpha=0.40, color="steelblue", label="Histogramme + KDE")
plt.plot(t, g, color="black", linestyle="--", lw=2,
         label="Densité théorique tronquée g")
plt.xlabel("x")
plt.ylabel("Densité")
plt.title("Normale N(0,1) tronquée sur [-1,1]")
plt.legend()
plt.show()
print("Nombre de valeurs conservées :", len(x_tronq_norm))

In [ ]:
# Exercice 2b — exponentielle de taux 1 tronquée sur [0.5, 2]
x_tronq_exp = []

...

x_tronq_exp = np.array(x_tronq_exp)

# Histogramme normalisé et courbe KDE (code fourni)
plt.figure(figsize=(7, 4))
sns.histplot(x_tronq_exp, bins=25, stat="density", kde=True,
             kde_kws={"cut": 0, "clip": (0.5, 2)},
             alpha=0.5, color="steelblue")
plt.xlabel("x")
plt.ylabel("Densité")
plt.title("Exponentielle de paramètre 1 tronquée sur [0.5,2]")
plt.show()
print("Nombre de valeurs conservées :", len(x_tronq_exp))

---
## Exercice 3 — Méthode d'inversion

Soit $U\sim\mathcal U(0,1)$ et $F$ une fonction de répartition **continue et strictement croissante** sur le support de la loi.

**a)** Montrer que $Z=F^{-1}(U)$ admet $F$ pour fonction de répartition.

**b)** Montrer que la fonction de répartition d'une loi **exponentielle de paramètre 2** est $F(t)=(1-e^{-2t})\mathbf 1_{\{t\geq0\}}$ et en déduire une **formule de simulation** à partir de $U$.

Rappel : Pour une variable aléatoire continue $X$ de densité $f$, la fonction de répartition est :
$
\boxed{F(x)=P(X\leq x)=\int_{-\infty}^{x}f(t)\,dt}
$

**c)** Simuler **1 000 valeurs par inversion** puis tracer son **histogramme normalisé et sa courbe KDE**.

In [ ]:
# Exercice 3c — compléter la formule d'inversion
u = ...
x_exp_inv = -np.log(1 - ...) / ...

t = np.linspace(0, max(4, x_exp_inv.max()), 350)
plt.figure(figsize=(7, 4))
sns.histplot(x_exp_inv, bins=30, stat="density", kde=True,
             kde_kws={"cut": 0, "clip": (0, np.inf)},
             alpha=0.40, color="steelblue", label="Histogramme + KDE")
plt.plot(t, 2 * np.exp(-2*t), color="black", linestyle="--",
         lw=2, label="Densité théorique")
plt.xlabel("x")
plt.ylabel("Densité")
plt.title("Exponentielle de paramètre 2 — inversion")
plt.legend()
plt.show()

---
## Exercice 5 — Simulation d’une loi jointe par conditionnement

### Données du problème

On souhaite simuler un échantillon de $1\,000$ couples $(X,V)$ de densité jointe

$$
\boxed{\displaystyle
f_{X,V}(x,v)=\frac{4}{\sqrt{2\pi}}\,v^{-9/2}
\exp\!\left(-\frac{(x-1)^2+4}{2v}\right)\mathbf{1}_{\{v>0\}},
\quad x\in\mathbb R.
}
$$

#### Une loi utile dans cet exo -- loi inverse-gamma 

On dit que $V$ suit une loi inverse-gamma $\mathrm{IG}(\alpha,\beta)$, pour $\alpha,\beta>0$, si sa densité est

$$
\boxed{\displaystyle
h(v)=\frac{\beta^{\alpha}}{\Gamma(\alpha)}\,v^{-\alpha-1} e^{-\beta/v}\,\mathbf 1_{\{v>0\}}.
}
$$

Ici, $\Gamma(3)=2$. Pour simuler cette loi avec SciPy :

```python
from scipy.stats import invgamma
v = stats.invgamma.rvs(a=alpha, scale=beta, size=n)
```

**Attention :** `stats.norm.rvs(loc=m, scale=s)` reçoit l'**écart-type** $s$, et non la variance.

### Partie A — Décomposition par conditionnement

**a)** Exprimer la densité marginale $f_V(v)$ comme une intégrale de la densité jointe. Calculer cette intégrale, puis identifier la loi de $V$ en utilisant le rappel inverse-gamma.

*Indication :* pour $v>0$,

$$
\int_{-\infty}^{+\infty}\frac{1}{\sqrt{2\pi v}}
\exp\!\left(-\frac{(x-1)^2}{2v}\right)\,dx=1.
$$

**b)** On rappelle que, pour tout $v$ tel que $f_V(v)>0$, la densité conditionnelle vérifie

$$
\boxed{\displaystyle
f_{X\mid V=v}(x)=\frac{f_{X,V}(x,v)}{f_V(v)}.
}
$$

À l'aide de cette relation et du résultat de la question a), **calculer** la densité de $X$ sachant $V=v$.

**c)** Identifier la loi conditionnelle de $X$ sachant $V=v$ et vérifier la factorisation

$$
f_{X,V}(x,v)=f_{X\mid V=v}(x)\,f_V(v).
$$


#### Interprétation pour ceux qui veulent: modèle bayésien de type normale–inverse-gamma

Cette décomposition conduit à un modèle bayésien de type normale–inverse-gamma à moyenne connue :

$$
\boxed{\begin{aligned}
\text{prior:}\quad &V\sim\mathrm{IG}(3,2),\\[3pt]
\text{vraisemblence:}\quad &X\mid(V=v)\sim\mathcal N(1,v).
\end{aligned}}
$$


**Principe de simulation par conditionnement.** Lorsqu'une densité jointe se factorise en **densité marginale × densité conditionnelle**, on peut simuler un couple en **tirant d'abord une valeur selon la marginale**, puis **en tirant la seconde variable selon la loi conditionnelle correspondant à cette valeur**.

**Algorithme à utiliser ici pour chaque couple $(X_i,V_i)$ :**

1. Tirer $V_i$ selon la loi inverse-gamma $\mathrm{IG}(3,2)$.
2. Une fois $V_i=v_i$ obtenu, tirer $X_i$ selon la loi normale $\mathcal N(1,v_i)$, **d'écart-type $\sqrt{v_i}$**.
3. Conserver le couple $(X_i,V_i)$.

Il suffit de répéter ces étapes pour obtenir un échantillon de couples issus de la loi jointe recherchée.

### Partie B — Simulation du couple et étude des marginales

**d)** À l'aide de l'algorithme donné dans la partie A, simuler un échantillon de $1\,000$ couples $(X,V)$. Représenter ensuite les couples simulés par un nuage de points, en superposant les lignes de niveau de la densité jointe estimée par la méthode KDE bidimensionnelle.

**e)** À partir des échantillons `x` et `v`, représenter pour chaque variable son **histogramme normalisé** ainsi qu'une **courbe KDE** calculée automatiquement avec `sns.histplot(..., stat="density", kde=True)`. 


**Rappels Python :**

```python
stats.invgamma.rvs(a=alpha, scale=beta, size=n)
stats.norm.rvs(loc=moyenne, scale=ecart_type, size=n)
sns.histplot(valeurs, stat="density", kde=True)
```

In [ ]:
# Questions d — compléter uniquement les ...
import numpy as np
from scipy import stats
n = 1000

# 1) Simuler les variances V selon IG(3, 2).
v = stats.invgamma.rvs(a=..., scale=..., size=n)

# 2) Simuler X | V = v selon N(1, v).
#    Attention : np.sqrt(...) donne l'écart-type correspondant.
x = stats.norm.rvs(loc=1, scale=np.sqrt(...), size=n)

# 3) Nuage de points
plt.figure(figsize=(8, 6))

plt.scatter(x, v, s=10, alpha=0.3,
            label="Couples simulés")

# 4) Lignes de niveau de la KDE bidimensionnelle
sns.kdeplot(x=x, y=v, levels=8,
            color="red", linewidths=1.5)

plt.xlabel("X")
plt.ylabel("V")
plt.title("Nuage de points et densité jointe estimée")
plt.legend()
plt.show()


In [ ]:
# Question e — histogrammes normalisés et KDE en une commande
import seaborn as sns
import matplotlib.pyplot as plt

def tracer_marginale(ax, valeurs, nom):
    sns.histplot(valeurs, bins=30, stat=..., kde=...,
                 alpha=0.40, color="steelblue", ax=ax)
    ax.set_title(f"Marginale de {nom}")
    ax.set_xlabel(nom)
    ax.set_ylabel("Densité")
    if nom == "V":
        # Limite d'affichage pour mieux voir le centre (queue droite très longue).
        ax.set_xlim(0, max(4, np.quantile(valeurs, 0.995)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
tracer_marginale(axes[0], x, "X")
tracer_marginale(axes[1], v, "V")
plt.show()

---
## Exercice 6 — Fonction de répartition empirique

**Rappel.** Pour une variable aléatoire $X$, la fonction de répartition théorique est $F(t)=\mathbb P(X\leq t)$. À partir d'un échantillon $x_1,\ldots,x_n$, on définit la **fonction de répartition empirique** :

$$
\boxed{F_n(t)=\frac1n\sum_{i=1}^n\mathbf 1_{\{x_i\le t\}}.}
$$

En **réutilisant l'échantillon `x_norm`** de la loi $\mathcal N(0,1)$ obtenu à l'exercice 1, compléter **les deux trous** du code suivant pour tracer $F_n$ et la superposer à la fonction de répartition théorique. **Commenter la comparaison.**

**Rappel Python :** `norm.cdf(t, loc=0, scale=1)` fournit la fonction de répartition normale théorique aux points `t`.

*Remarque :* on utilise le nom `x_norm` pour éviter de confondre cet échantillon avec l'échantillon `x` du couple simulé à l'exercice 5.

In [ ]:
# Exercice 6 — compléter uniquement les deux ...
def F_empirique(t, x):
    return np.mean(x <= ...)  # À compléter

grille = np.linspace(-3, 3, 300)
F_n = [F_empirique(t, x_norm) for t in ...]  # À compléter

plt.figure(figsize=(7, 4))
plt.step(grille, F_n, where="post", label="Empirique")
plt.plot(grille, norm.cdf(grille, loc=0, scale=1),
         label="Théorique")
plt.xlabel("t")
plt.ylabel("F(t)")
plt.title("Fonction de répartition empirique et théorique")
plt.legend()
plt.show()

### Pour aller plus loin — Autre exemple de simulation en deux étapes *(facultatif)*

On considère un couple $(A,B)$ de densité jointe

$$
f_{A,B}(a,b) = 2\,\mathbf 1_{\{0<b<a<1\}}.
$$

**1.** Calculer la densité marginale $f_A$ et la densité conditionnelle $f_{B\mid A=a}$. Identifier la loi de $B$ sachant $A=a$.

**2.** Montrer que $F_A(a)=a^2$ pour $0\leq a\leq1$. En déduire une simulation à partir de deux uniformes indépendantes $U,W$ en posant $A=\sqrt U$ et $B=A W$.

**3.** Simuler $1\,000$ couples et représenter leur nuage. Décrire la région occupée.

---
### Documentation Python (pour les commandes utilisées)

- [Seaborn — explication pédagogique de la méthode KDE](https://seaborn.pydata.org/tutorial/distributions.html#kernel-density-estimation)
- [SciPy — tutoriel sur l’estimation de densité par noyau](https://docs.scipy.org/doc/scipy/tutorial/stats/kernel_density_estimation.html)
- [Seaborn — `histplot` : histogramme normalisé et KDE](https://seaborn.pydata.org/generated/seaborn.histplot.html)
- [SciPy — loi normale `norm`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [SciPy — loi inverse-gamma `invgamma`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.invgamma.html)

**Fin du TP 3.**